In [2]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import datetime as dt
import pymysql
import geopandas as gpd

# Table 하나씩 불러오기

In [3]:
# Call reservation request table
con = pymysql.connect(
    user='root',
    passwd='3644',
    host='143.248.121.90',
    port=3306,
    db='hdl',
    charset='utf8mb4',
    use_unicode=True
)

mycursor = con.cursor()
query = """
    select * from reservation_request;
"""
mycursor.execute(query)
data = mycursor.fetchall()
con.close()
request_df = pd.DataFrame(data, columns=["requestID", "passengerID", "messageTime", "pickupStationID", "dropoffStationID", "serviceType", "reserveType", "dispatchID", "responseStatus", "confirmCheck", "passengerCount", "wheelchairCount", "failInfoList", "pickupTimeRequest"])

In [5]:
# Call dispatch table
con = pymysql.connect(
    user='root',
    passwd='3644',
    host='143.248.121.90',
    port=3306,
    db='hdl',
    charset='utf8mb4',
    use_unicode=True
)

mycursor = con.cursor()
query = """
    select * from dispatch;
"""
mycursor.execute(query)
data = mycursor.fetchall()
con.close()
dispatch_df = pd.DataFrame(data, columns=["dispatchID", "messageTime", "passengerID", "requestID", "routeIDs", "pickupStationName", "dropoffStationName", "reserveType", "onboardingTime", "dropoffTime", "linkIDs", "pickupStationID", "dropoffStationID", "tripID", "operationID", "vehicleID"])

In [7]:
# Call operation table
con = pymysql.connect(
    user='root',
    passwd='3644',
    host='143.248.121.90',
    port=3306,
    db='hdl',
    charset='utf8mb4',
    use_unicode=True
)

mycursor = con.cursor()
query = """
    select * from operation;
"""
mycursor.execute(query)
data = mycursor.fetchall()
con.close()
operation_df = pd.DataFrame(data, columns=["operationID", "vehicleID", "StationIDs", "routeIDs", "startTime", "endTime", "VehicleType", "operationServiceType"])

In [9]:
# Call route table
con = pymysql.connect(
    user='root',
    passwd='3644',
    host='143.248.121.90',
    port=3306,
    db='hdl',
    charset='utf8mb4',
    use_unicode=True
)

mycursor = con.cursor()
query = """
    select * from route;
"""
mycursor.execute(query)
data = mycursor.fetchall()
con.close()
route_df = pd.DataFrame(data, columns=["routeID", "routeSeq", "operationID", "vehicleID", "routeInfo", "linkIDs", "NodeIDs", "originStationID", "originDeptTime", "destinationID", "onboardingNum", "dispatchIDs", "lon", "lat", "originBoardingPxIDs", "originGetoffPxIDs", "destBoardingPxIDs", "destGetoffPxIDs", "destArrivalTime", "routeCode"])

# 불러온 데이터확인

In [10]:
request_df

,requestID,passengerID,messageTime,pickupStationID,dropoffStationID,serviceType,reserveType,dispatchID,responseStatus,confirmCheck,passengerCount,wheelchairCount,failInfoList,pickupTimeRequest
0,req-0001,229,1768432263000,S233002900,S233000180,2,2,D38353a1ec,0.0,1.0,1,1,None,NaN
1,req-0002,638,1768433048000,S233004400,S233004190,1,1,D4f202e842,0.0,1.0,1,1,None,1130.0
2,req-0003,171,1768433161000,S233003070,S233001790,1,2,D1ec6e5aa4,0.0,1.0,2,1,None,NaN
3,req-0004,282,1768434844000,S233001880,S233001320,1,1,Dbeb945766,0.0,1.0,2,1,None,2030.0
4,req-0005,616,1767916087000,S233004520,S233001590,1,1,D248f59b29,0.0,1.0,3,0,None,1930.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1483,req-7064,521,1772514608000,S233006810,S233006770,2,1,D9cc60810e,0.0,1.0,3,0,None,1500.0
1484,req-7067,352,1772516102000,S233004360,S233008710,1,1,D64b5a06de,0.0,1.0,1,0,None,1830.0
1485,req-7070,669,1772517187000,S233007000,S233007830,2,1,Dce3a3ea21,0.0,1.0,2,0,None,1930.0
1486,req-7076,977,1772519225000,S233001320,S233002360,1,1,D642b9627e,0.0,1.0,1,0,None,1830.0


In [11]:
dispatch_df

,dispatchID,messageTime,passengerID,requestID,routeIDs,pickupStationName,dropoffStationName,reserveType,onboardingTime,dropoffTime,linkIDs,pickupStationID,dropoffStationID,tripID,operationID,vehicleID
0,D008e8e8ef,1771038963063,648,req-5047,[2026021500032],원천교회,No Station Name,1,202602151400,2.026022e+11,"[""L233010460"", ""L233008780"", ""L233008740"", ""L2...",S233007850,S233006860,Tae0bf5b34,202602140008,TW_000000006
1,D00c207127,1769987829948,160,req-3701,[2026020300005],경기수협보건소,No Station Name,1,202602031900,2.026020e+11,"[""L233002080"", ""L233002090"", ""L233002180"", ""L2...",S233004800,S233006700,T75f2d66ed,202602020001,TW_000000006
2,D00dd8ab1f,1771731004786,880,req-6147,[2026022300017],화성시청후문,No Station Name,1,202602231300,2.026022e+11,"[""L233000540"", ""L233000550"", ""L233000860"", ""L2...",S233005660,S233004400,Tf2092d8dd,202602220004,TW_000000001
3,D01019066e,1771988470536,183,req-6545,[2026022600044],No Station Name,No Station Name,1,202602261430,2.026023e+11,"[""L233002470"", ""L233002480"", ""L233002950"", ""L2...",S233004990,S233001390,T57360f9c3,202602250010,TW_000000001
4,D019c8fd34,1771488068149,195,req-5786,[2026022000044],No Station Name,No Station Name,1,202602202133,2.026022e+11,"[""L233013330"", ""L233013340"", ""L233014090"", ""L2...",S233002640,S233004320,Tbb5f1271f,202602190009,TW_000000001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,Dfee4d938a,1768179369185,690,req-0439,[2026011300026],No Station Name,No Station Name,1,202601121340,2.026011e+11,"[""L233001490"", ""L233001500"", ""L233001510"", ""L2...",S233004570,S233004510,T990d4cb23,202601110005,TW_000000001
1015,Dff01d5b41,1772001849433,936,req-6581,[2026022600062],No Station Name,No Station Name,1,202602261900,2.026023e+11,"[""L233020800"", ""L233008670"", ""L233008690"", ""L2...",S233000450,S233007770,Ta3d27429c,202602250008,TW_000000006
1016,Dff1854a4d,1768116242589,459,req-0357,[2026011200076],No Station Name,No Station Name,1,202601121734,2.026011e+11,"[""L233020770"", ""L233022000"", ""L233025540"", ""L2...",S233000490,S233007800,T6aa68fa4f,202601110005,TW_000000006
1017,Dff4bda03e,1771215791244,592,req-5330,[2026021700053],No Station Name,No Station Name,1,202602171630,2.026022e+11,"[""L233001270"", ""L233001280"", ""L233001290"", ""L2...",S233004380,S233004520,Tdc202008d,202602160008,TW_000000001


In [12]:
operation_df

,operationID,vehicleID,StationIDs,routeIDs,startTime,endTime,VehicleType,operationServiceType
0,202601090001,TW_000000001,"[""S233002090"", ""S233008810"", ""S233001280"", ""S2...","[2026011000001, 2026011000002, 2026011000003]",202601102023,202601102043,carnivalReg,POOL
1,202601090001,TW_000000004,"[""S233002090"", ""S233004050"", ""S233003310"", ""S2...","[2026010900001, 2026010900002, 2026010900004, ...",202601090911,202601091010,carnivalWheel,P2P
2,202601090001,TW_000000005,"[""S233002090"", ""S233004510"", ""S233002260"", ""S2...","[2026011000019, 2026011000020, 2026011000021]",202601101059,202601101112,carnivalWheel,P2P
3,202601090001,TW_000000006,"[""S233007690"", ""S233008100"", ""S233007100"", ""S2...","[2026011000004, 2026011000005, 2026011000006]",202601101742,202601101919,IONIQ5,POOL
4,202601090001,TW_000000007,"[""S233007690"", ""S233000660"", ""S233000900""]","[2026011000043, 2026011000044]",202601102028,202601102059,IONIQ5,P2P
...,...,...,...,...,...,...,...,...
899,202603030009,TW_000000006,"[""S233007690"", ""S233007000"", ""S233007830"", ""S2...","[2026030400058, 2026030400059, 2026030400060]",202603041906,202603042003,IONIQ5,POOL
900,202603030010,TW_000000001,"[""S233002090"", ""S233002580"", ""S233002360"", ""S2...","[2026030400040, 2026030400041, 2026030400042]",202603041530,202603041542,carnivalReg,POOL
901,202603030011,TW_000000001,"[""S233002090"", ""S233004360"", ""S233008710"", ""S2...","[2026030400055, 2026030400056, 2026030400057]",202603041823,202603041841,carnivalReg,POOL
902,202603030012,TW_000000001,"[""S233002090"", ""S233001320"", ""S233002360"", ""S2...","[2026030400061, 2026030400062, 2026030400063]",202603041822,202603041839,carnivalReg,POOL


In [13]:
route_df

,routeID,routeSeq,operationID,vehicleID,routeInfo,linkIDs,NodeIDs,originStationID,originDeptTime,destinationID,onboardingNum,dispatchIDs,lon,lat,originBoardingPxIDs,originGetoffPxIDs,destBoardingPxIDs,destGetoffPxIDs,destArrivalTime,routeCode
0,2026010900001,0.0,202601090001,TW_000000004,0.0,"[""L233015001"", ""L233015010"", ""L233015020"", ""L2...","[""S233002090"", ""N233017050"", ""N233017040"", ""N2...",S233002090,2.026011e+11,S233004050,0.0,[],"[126.833601, 126.833851, 126.8341695, 126.8346...","[37.2093538, 37.2093906, 37.2094384, 37.209472...",[],[948],[599],[],2.026011e+11,1.0
1,2026010900002,1.0,202601090001,TW_000000004,1.0,"[""L233015910"", ""L233015920"", ""L233020430"", ""L2...","[""S233004050"", ""N233013100"", ""N233013840"", ""N2...",S233004050,2.026011e+11,S233003310,2.0,"[""D181ee6485""]","[126.8270895, 126.8270087, 126.8269744, 126.82...","[37.1970681, 37.1971464, 37.1971795, 37.197336...",[599],[],[],[599],2.026011e+11,1.0
2,2026010900004,2.0,202601090001,TW_000000004,1.0,"[""L233022120"", ""L233022130"", ""L233022230"", ""L2...","[""S233003310"", ""N233020280"", ""N233019170"", ""N2...",S233003310,2.026011e+11,S233005020,0.0,[],"[126.824417, 126.8251525, 126.8257406, 126.825...","[37.2065045, 37.2065811, 37.2072139, 37.207262...",[],[599],[512],[],2.026011e+11,1.0
3,2026010900005,3.0,202601090001,TW_000000004,1.0,"[""L233003670"", ""L233003680"", ""L233003690"", ""L2...","[""S233005020"", ""N233008160"", ""N233009660"", ""N2...",S233005020,2.026011e+11,S233002730,1.0,"[""D0939ed9fb""]","[126.8258284, 126.8255419, 126.8254116, 126.82...","[37.2104663, 37.2109725, 37.2111746, 37.211524...",[512],[],[],[512],2.026011e+11,1.0
4,2026010900007,0.0,202601090009,TW_000000006,0.0,"[""L233010760"", ""L233010770"", ""L233009260"", ""L2...","[""S233007690"", ""N233003900"", ""N233003910"", ""N2...",S233007690,2.026011e+11,S233000420,0.0,[],"[126.8026311, 126.8026359, 126.8026392, 126.80...","[37.2406814, 37.2406256, 37.2405869, 37.238080...",[],[],[557],[],2.026011e+11,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2916,2026030400062,1.0,202603030012,TW_000000001,1.0,"[""L233019630"", ""L233019640"", ""L233019641"", ""L2...","[""S233001320"", ""N233013180"", ""N233021330"", ""N2...",S233001320,2.026030e+11,S233002360,1.0,"[""D642b9627e""]","[126.8283734, 126.8280203, 126.8278458, 126.82...","[37.1983308, 37.1986728, 37.1988416, 37.199048...",[977],[],[],[977],2.026030e+11,1.0
2917,2026030400063,2.0,202603030012,TW_000000001,2.0,"[""L233014200"", ""L233014020"", ""L233014010"", ""L2...","[""S233002360"", ""N233015470"", ""N233015480"", ""N2...",S233002360,2.026030e+11,S233002090,0.0,[],"[126.8368122, 126.8361959, 126.8361633, 126.83...","[37.2066722, 37.2065203, 37.2065161, 37.206490...",[],[977],[],[],2.026030e+11,1.0
2918,2026030400064,0.0,202603030013,TW_000000001,0.0,"[""L233015001"", ""L233015010"", ""L233015020"", ""L2...","[""S233002090"", ""N233017050"", ""N233017040"", ""N2...",S233002090,2.026030e+11,S233002220,0.0,[],"[126.833601, 126.833851, 126.8341695, 126.8346...","[37.2093538, 37.2093906, 37.2094384, 37.209472...",[],[],[548],[],2.026030e+11,1.0
2919,2026030400065,1.0,202603030013,TW_000000001,1.0,"[""L233013290"", ""L233013360"", ""L233013370"", ""L2...","[""S233002220"", ""N233015260"", ""N233015350"", ""N2...",S233002220,2.026030e+11,S233004540,2.0,"[""D08c4d1860""]","[126.8312155, 126.8313594, 126.8313903, 126.83...","[37.2056209, 37.2056374, 37.2056409, 37.205648...",[548],[],[],[548],2.026030e+11,1.0


# 여기서부터 하나씩 나머지 코드 실행

In [ ]:
# for example, P1_sevice_arrival_operation_times.py

def parse_onboarding_time(t):
    try:
        t_str = str(int(t)).zfill(12)
        return datetime.strptime(t_str, "%Y%m%d%H%M")
    except:
        return np.nan
    
dispatch_df['onboarding_datetime'] = dispatch_df['onboardingTime'].apply(parse_onboarding_time)
dispatch_df['dropoff_datetime'] = dispatch_df['dropoffTime'].apply(parse_onboarding_time)

operation_df['startTime_datetime'] = operation_df['startTime'].apply(parse_onboarding_time)
operation_df['endTime_datetime'] = operation_df['endTime'].apply(parse_onboarding_time)

route_df['originDeptTime_datetime'] = route_df['originDeptTime'].apply(parse_onboarding_time)
route_df['destArrivalTime_datetime'] = route_df['destArrivalTime'].apply(parse_onboarding_time)

# ...